# Lab W6: Temporal Leakage Demonstration


**Pendamping Bab W6.**

Tujuan: mendemonstrasikan bagaimana **temporal leakage** - penggunaan informasi dari masa depan secara tidak sengaja - bisa menginflasi metrik secara dramatis.

Kita bandingkan dua pipeline:
1. **Causal**: rolling features hanya dari timestep sebelumnya, split chronological
2. **Leaky**: random split + rolling features tanpa guard temporal (bocor masa depan)

**Prasyarat:** familiarity dengan time series dan classification.
**Estimasi:** 2-3 jam.


## 1. Setup

Synthetic dataset: slow-moving sinusoidal trend + noise, dengan label biner berdasarkan future median.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

np.random.seed(42)

def generate_temporal_dataset(n_samples=2000, noise=0.3):
    """Generate time series with temporal dependency.
    Target: 1 if current value > rolling median of FUTURE 20 values.
    """
    t = np.linspace(0, 40 * np.pi, n_samples)
    trend = np.sin(t) * 0.5 + np.sin(t * 0.15) * 0.3
    value = trend + np.random.randn(n_samples) * noise

    # Label: future-looking (THIS IS THE LEAK)
    future_median = pd.Series(value).rolling(20, min_periods=1).median().shift(-10).values
    y = (value > future_median).astype(int)

    df = pd.DataFrame({"value": value, "trend": trend, "y": y})
    return df

df = generate_temporal_dataset(2000, noise=0.3)
print(f"Shape: {df.shape}")
print(f"Class ratio: {df.y.mean():.3f} / {1-df.y.mean():.3f}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
ax1.plot(df["trend"], label="True trend", alpha=0.7)
ax1.plot(df["value"], label="Observed value", alpha=0.4)
ax1.set_title("Time Series (first 300 points)")
ax1.set_xlabel("Time"); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_xlim(0, 300)

ax2.scatter(range(len(df)), df["y"], s=5, c=df["y"], cmap="coolwarm", alpha=0.5)
ax2.set_title("Label (red=1, blue=0) - first 300 points")
ax2.set_xlabel("Time"); ax2.set_ylabel("Class")
ax2.set_xlim(0, 300)
plt.tight_layout(); plt.show()

## 2. Feature Engineering

Buat rolling features. Pipeline causal: shift sebelum rolling (hanya masa lalu). Pipeline leaky: rolling langsung (bocor masa depan).


In [ ]:
def make_features_causal(df, window=10):
    """Rolling features: hanya dari masa lalu (shift 1)."""
    feat = pd.DataFrame(index=df.index)
    feat["value_lag1"] = df["value"].shift(1)
    feat["value_lag3"] = df["value"].shift(3)
    feat["value_lag5"] = df["value"].shift(5)
    feat["roll_mean"] = df["value"].shift(1).rolling(window, min_periods=3).mean()
    feat["roll_std"] = df["value"].shift(1).rolling(window, min_periods=3).std()
    feat["roll_min"] = df["value"].shift(1).rolling(window, min_periods=3).min()
    feat["roll_max"] = df["value"].shift(1).rolling(window, min_periods=3).max()
    feat["diff1"] = df["value"].diff(1)
    feat["diff3"] = df["value"].diff(3)
    feat["trend"] = df["trend"]
    return feat

def make_features_leaky(df, window=10):
    """Rolling features: TANPA shift - bocor informasi masa depan!"""
    feat = pd.DataFrame(index=df.index)
    feat["value_lag1"] = df["value"].shift(1)
    feat["roll_mean"] = df["value"].rolling(window, min_periods=3).mean()
    feat["roll_std"] = df["value"].rolling(window, min_periods=3).std()
    feat["roll_min"] = df["value"].rolling(window, min_periods=3).min()
    feat["roll_max"] = df["value"].rolling(window, min_periods=3).max()
    feat["future_mean"] = df["value"].shift(-5).rolling(5, min_periods=2).mean()
    feat["diff1"] = df["value"].diff(1)
    feat["trend"] = df["trend"]
    return feat

feat_causal = make_features_causal(df)
feat_leaky = make_features_leaky(df)

print(f"Causal features: {feat_causal.shape[1]} cols, {feat_causal.isna().sum().sum()} NaN")
print(f"Leaky features:  {feat_leaky.shape[1]} cols, {feat_leaky.isna().sum().sum()} NaN")
print("\nCausal feature columns:")
for c in feat_causal.columns:
    print(f"  {c}")
print("\nLeaky feature columns (extra leakage):")
leaky_extra = set(feat_leaky.columns) - set(feat_causal.columns)
for c in feat_leaky.columns:
    if c in leaky_extra:
        print(f"  [LEAK] {c}")

## 3. Causal Pipeline

Split data **chronologically**: 80% pertama train, 20% terakhir test. Drop NaN. Train RandomForest.


In [ ]:
def chronological_split(X, y, ratio=0.8):
    n = len(X)
    split = int(n * ratio)
    return X[:split], X[split:], y[:split], y[split:]

X_c = feat_causal.dropna().values
y_c = df.loc[feat_causal.dropna().index, "y"].values
Xc_tr, Xc_te, yc_tr, yc_te = chronological_split(X_c, y_c)

clf_c = RandomForestClassifier(n_estimators=100, random_state=42)
clf_c.fit(Xc_tr, yc_tr)
pred_c = clf_c.predict(Xc_te)

acc_c = accuracy_score(yc_te, pred_c)
f1_c = f1_score(yc_te, pred_c)
print(f"=== CAUSAL PIPELINE === (split: chronological)")
print(f"Train size: {len(Xc_tr)}, Test size: {len(Xc_te)}")
print(f"Accuracy: {acc_c:.4f}")
print(f"F1 Score: {f1_c:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
cm_c = confusion_matrix(yc_te, pred_c)
ConfusionMatrixDisplay(cm_c).plot(ax=ax)
ax.set_title("Causal Pipeline - Confusion Matrix")
plt.tight_layout(); plt.show()

## 4. Leaky Pipeline

Split data **randomly** (shuffle). Rolling features TANPA shift. Ini kesalahan klasik: split sebelum feature engineering.


In [ ]:
X_l = feat_leaky.dropna().values
y_l = df.loc[feat_leaky.dropna().index, "y"].values

Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(
    X_l, y_l, test_size=0.2, random_state=42, shuffle=True
)

clf_l = RandomForestClassifier(n_estimators=100, random_state=42)
clf_l.fit(Xl_tr, yl_tr)
pred_l = clf_l.predict(Xl_te)

acc_l = accuracy_score(yl_te, pred_l)
f1_l = f1_score(yl_te, pred_l)
print(f"=== LEAKY PIPELINE === (split: random shuffle)")
print(f"Train size: {len(Xl_tr)}, Test size: {len(Xl_te)}")
print(f"Accuracy: {acc_l:.4f}")
print(f"F1 Score: {f1_l:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
cm_l = confusion_matrix(yl_te, pred_l)
ConfusionMatrixDisplay(cm_l).plot(ax=ax)
ax.set_title("Leaky Pipeline - Confusion Matrix")
plt.tight_layout(); plt.show()

## 5. Leakage Inflation

Hitung Leakage Inflation = F1_leaky - F1_causal. Semakin besar, semakin berbahaya.


In [ ]:
print("=== LEAKAGE INFLATION REPORT ===")
print(f"\nPipeline          Accuracy     F1")
print(f"---               --------     --")
print(f"Causal  (chrono)  {acc_c:.4f}      {f1_c:.4f}")
print(f"Leaky   (random)  {acc_l:.4f}      {f1_l:.4f}")
print(f"---")
print(f"Inflation        {acc_l-acc_c:+.4f}      {f1_l-f1_c:+.4f}")

inflation = f1_l - f1_c
print(f"\n=== LEAKAGE INFLATION = {inflation:.4f} ===")

if inflation > 0.1:
    print(">>> PERINGATAN: Inflasi > 0.1! Temporal leakage mendominasi performa.")
elif inflation > 0.05:
    print(">>> WASPADA: Inflasi moderat. Ada leakage yang perlu diperbaiki.")
else:
    print(">>> Inflasi kecil atau tidak ada. Pipeline cukup aman.")

feature_names = list(feat_leaky.dropna().columns)
importances = clf_l.feature_importances_
top_idx = np.argsort(importances)[-5:]
print("\nTop-5 fitur paling penting di leaky pipeline:")
for i in top_idx:
    leak_flag = "[LEAK]" if "roll_" in feature_names[i] or "future" in feature_names[i] else ""
    print(f"  {feature_names[i]:20s} {importances[i]:.4f}  {leak_flag}")

## 6. Diagnostik

Plot prediksi leaky vs causal pada test set. Leaky biasanya menang di titik perubahan trend karena sudah 'melihat' masa depan.


In [ ]:
idx_causal = feat_causal.dropna().index[-len(pred_c):]
df_causal = df.loc[idx_causal].copy()
df_causal["pred"] = pred_c

idx_leaky = feat_leaky.dropna().index[-len(pred_l):]
df_leaky = df.loc[idx_leaky].copy()
df_leaky["pred"] = pred_l

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df_causal.index, df_causal["y"], label="True label", color="black", alpha=0.7)
axes[0].set_ylabel("True label")
axes[0].set_title("Ground Truth")
axes[0].grid(alpha=0.3)

correct_c = df_causal["y"] == df_causal["pred"]
axes[1].plot(df_causal.index, df_causal["pred"], label="Causal pred", color="blue", alpha=0.7)
axes[1].fill_between(df_causal.index, 0, 1, where=~correct_c, color="red", alpha=0.2, label="Error")
axes[1].set_ylabel("Causal pred")
axes[1].set_title(f"Causal Pipeline (F1={f1_c:.3f})")
axes[1].grid(alpha=0.3); axes[1].legend(loc="upper right")

correct_l = df_leaky["y"] == df_leaky["pred"]
axes[2].plot(df_leaky.index, df_leaky["pred"], label="Leaky pred", color="red", alpha=0.7)
axes[2].fill_between(df_leaky.index, 0, 1, where=~correct_l, color="red", alpha=0.15, label="Error")
axes[2].set_ylabel("Leaky pred")
axes[2].set_xlabel("Time")
axes[2].set_title(f"Leaky Pipeline (F1={f1_l:.3f})")
axes[2].grid(alpha=0.3); axes[2].legend(loc="upper right")

plt.tight_layout(); plt.show()

print("Apa pola yang terlihat? Leaky pipeline biasanya lebih akurat di sekitar")
print("titik perubahan trend, karena rolling window-nya 'mengintip' masa depan.")

## 7. Kenapa Ini Berbahaya?

Model leaky mendapat F1 tinggi di validasi. Di deploy, performa anjlok karena tidak ada data masa depan. Rolling window yang waktu training melihat ke depan, sekarang hanya bisa melihat ke belakang.

**Moral:** accuracy/F1 tinggi di hari pertama bukan kabar baik - itu lampu merah.

### Tugas:
Tulis satu paragraf: apa yang membuat angka leaky terlihat meyakinkan, dan mengapa tetap salah?


In [ ]:
# Tulis jawaban Anda di sel markdown di atas.
print("Sel sengaja kosong.")

## 8. Refleksi

1. **Warp zone.** Coba ulangi dengan `window=3` vs `window=50`. Apakah leakage inflation berubah? Kenapa?

2. **Deteksi dini.** Dari 5-layer EDA di W6, layer mana yang pertama menangkap temporal leakage?

3. **Koneksi ke Capstone.** Dataset Capstone Anda mungkin punya komponen temporal. Tuliskan cara memverifikasi tidak ada temporal leakage di pipeline Capstone Anda.
